# IOAI — 2026 Regional 11 12 Nurse (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, zipfile, urllib.request
os.makedirs('data', exist_ok=True)
if not os.path.exists('data/train_data.csv'):
    urllib.request.urlretrieve('https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/roai-nurse.zip', '/tmp/nurse.zip')
    with zipfile.ZipFile('/tmp/nurse.zip') as zf: zf.extractall('data')
print('데이터 준비:', 'data/train_data.csv' if os.path.exists('data/train_data.csv') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# The Good Nurse — ICU 28일 사망 예측 (RoAI 2026 Regional 11-12)

ICU 임상 표 데이터로 3 서브태스크:
1. (결정적) `day_28_flg` 결측 개수
2. (결정적) SOFA>10 vs ≤10 그룹 사망률(%)
3. (ML) `day_28_flg` 사망 **확률** 예측 — ROC-AUC 로 채점

**제출**: `submission.csv` 롱포맷 `datapointID, subtaskID, answer`.
**채점(로컬)**: ST1/2 수치 근사, ST3 train held-out ROC-AUC. **데이터**: `data/train_data.csv`(+`data/test_data.csv`).
원본: judge.nitro-ai.org/competitions (ROAI etapa-judeteana XI-XII)

In [ ]:
import pandas as pd, numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

root_path = "data"
df_train = pd.read_csv(f"{root_path}/train_data.csv")
df_test = pd.read_csv(f"{root_path}/test_data.csv")
df_train.head()

In [ ]:
# ST1 (결정적): day_28_flg 결측 개수
subtask1 = [int(df_train["day_28_flg"].isna().sum())]
# ST2 (결정적): SOFA>10 vs <=10 그룹 사망률(%)
lab = df_train.dropna(subset=["day_28_flg"])
mort_gt10 = lab[lab["sofa_first"]>10]["day_28_flg"].mean()*100
mort_le10 = lab[lab["sofa_first"]<=10]["day_28_flg"].mean()*100
subtask2 = [mort_gt10, mort_le10]

In [ ]:
# ST3 (ML, 채점 대상): 사망확률 — 간단 베이스라인(라벨 있는 행만 LogReg).
# 라벨이 100개뿐이고 미라벨이 1235개 → 모범답안은 KNN 대치로 미라벨을 활용(준지도)해 AUC 를 올립니다.
feat = [c for c in df_train.select_dtypes(include=np.number).columns if c != "day_28_flg" and c in df_test.columns]
clf = make_pipeline(SimpleImputer(), StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced"))
clf.fit(lab[feat], lab["day_28_flg"].astype(int))
subtask3 = clf.predict_proba(df_test[feat])[:, 1]

In [ ]:
# 제출(롱포맷): datapointID, subtaskID, answer
def build_submission(sid, ans):
    id_ = df_test["ID"] if sid == 3 else (1 if sid == 1 else [1, 2])
    return pd.DataFrame({"datapointID": id_, "subtaskID": sid, "answer": ans})
sub = pd.concat([build_submission(1, subtask1), build_submission(2, subtask2), build_submission(3, subtask3)])
sub.to_csv(f"{root_path}/submission.csv", index=False)
sub.head()

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)